In [1]:
import xarray as xr
import gsw
import cartopy.crs as ccrs
import cartopy.feature as cfeature  
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import glob
import dask.array as da

LOAD IN SITU AND BATHYMETRY FILE

In [2]:
out_path = '../data_for_lstm/var_depths_data_for_LSTM_B3_bathy.nc'

In [3]:
ds_insitu = xr.open_dataset('../data_for_lstm/var_depths_data_for_LSTM_B2_wgm.nc',
                            decode_times=True)
ds_insitu

<xarray.Dataset> Size: 2GB
Dimensions:            (profile: 183565, depth: 102)
Coordinates:
  * profile            (profile) int64 1MB 0 1 2 3 ... 183562 183563 183564
  * depth              (depth) float64 816B 0.0 5.0 10.0 ... 5.4e+03 5.5e+03
Data variables: (12/23)
    LATITUDE           (profile) float64 1MB ...
    LONGITUDE          (profile) float64 1MB ...
    TIME               (profile) datetime64[ns] 1MB ...
    TEMP               (profile, depth) float64 150MB ...
    PSAL               (profile, depth) float64 150MB ...
    PRES               (profile, depth) float64 150MB ...
    ...                 ...
    SST                (profile) float64 1MB ...
    SSS                (profile) float64 1MB ...
    S_glorys           (profile, depth) float64 150MB ...
    T_glorys           (profile, depth) float64 150MB ...
    S_glorys_monthly   (profile, depth) float64 150MB ...
    T_glorys_monthly   (profile, depth) float64 150MB ...
Attributes:
    title:                  An Arctic Ocean Thermohaline Dataset
    institution:            Key Laboratory of Marine Hazard Forecasting, Mini...
    platform_type:          XX
    doi:                    10.1038/s41597-025-05855-3
    glorys_offset_pattern:  A tolerance window of 5 by 5 indices centered at ...

In [4]:

ds_bathy = xr.open_dataset('/home/nicolas/SACO/FRESH-CARE/Data_bathymetry/GEBCO_Sep_2025/data_EASE/gebco_ease_grid_1km.nc')
ds_bathy

<xarray.Dataset> Size: 612MB
Dimensions:            (y_ease: 8744, x_ease: 8744)
Coordinates:
  * y_ease             (y_ease) int64 70kB -4372000 -4371000 ... 4370000 4371000
  * x_ease             (x_ease) int64 70kB -4372000 -4371000 ... 4370000 4371000
Data variables:
    elevation          (y_ease, x_ease) float64 612MB ...
    ease_grid_mapping  int64 8B ...
Attributes:
    title:               GEBCO Elevation/Bathymetry aggregated to EASE Grid
    source:              Aggregated from data/gebco_2025_sub_ice_n90.0_s50.0_...
    grid_resolution:     1.0 km
    projection:          Lambert Azimuthal Equal Area (Arctic)
    description:         Representative elevation in each EASE grid cell. Max...
    aggregation_method:  Maximum elevation for land, minimum absolute depth f...
    creation_date:       2025-11-26
    conventions:         CF-1.8

ADD BATHYMETRY AT EACH PROFILE

In [5]:
interp_bathy = ds_bathy.elevation.interp(
    coords={'x_ease': ds_insitu.X_EASE,
            'y_ease': ds_insitu.Y_EASE},
            method='linear'
    )

In [6]:
interp_bathy

<xarray.DataArray 'elevation' (profile: 183565)> Size: 1MB
array([-3831.967508,   -92.87424 ,  -172.121472, ...,  -305.16768 ,
       -2369.51067 ,  -352.3074  ], shape=(183565,))
Coordinates:
  * profile  (profile) int64 1MB 0 1 2 3 4 ... 183561 183562 183563 183564
    x_ease   (profile) float64 1MB -6.494e+05 2.643e+05 ... -4.477e+05
    y_ease   (profile) float64 1MB 1.122e+06 -1.324e+06 ... -2.104e+06
Attributes:
    long_name:     Representative elevation in EASE grid cell
    units:         m
    description:   Maximum elevation for land areas (>0), minimum absolute de...
    grid_mapping:  ease_grid_mapping

In [7]:
ds_insitu['bathymetry'] = interp_bathy
ds_insitu

<xarray.Dataset> Size: 2GB
Dimensions:            (profile: 183565, depth: 102)
Coordinates:
  * profile            (profile) int64 1MB 0 1 2 3 ... 183562 183563 183564
    x_ease             (profile) float64 1MB -6.494e+05 2.643e+05 ... -4.477e+05
    y_ease             (profile) float64 1MB 1.122e+06 -1.324e+06 ... -2.104e+06
  * depth              (depth) float64 816B 0.0 5.0 10.0 ... 5.4e+03 5.5e+03
Data variables: (12/24)
    LATITUDE           (profile) float64 1MB ...
    LONGITUDE          (profile) float64 1MB ...
    TIME               (profile) datetime64[ns] 1MB ...
    TEMP               (profile, depth) float64 150MB ...
    PSAL               (profile, depth) float64 150MB ...
    PRES               (profile, depth) float64 150MB ...
    ...                 ...
    SSS                (profile) float64 1MB ...
    S_glorys           (profile, depth) float64 150MB ...
    T_glorys           (profile, depth) float64 150MB ...
    S_glorys_monthly   (profile, depth) float64 150MB ...
    T_glorys_monthly   (profile, depth) float64 150MB ...
    bathymetry         (profile) float64 1MB -3.832e+03 -92.87 ... -352.3
Attributes:
    title:                  An Arctic Ocean Thermohaline Dataset
    institution:            Key Laboratory of Marine Hazard Forecasting, Mini...
    platform_type:          XX
    doi:                    10.1038/s41597-025-05855-3
    glorys_offset_pattern:  A tolerance window of 5 by 5 indices centered at ...

In [8]:
ds_insitu.to_netcdf(out_path)